In [ ]:
import pandas as pd
import yfinance as yfin
import datetime as dt
import numpy as np
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
Ticker = ['^NSEI']

# Indicate the start and end dates
start = dt.datetime.now().replace(year=dt.datetime.now().year - 1)
end = dt.datetime.now()

nifty = yfin.download(Ticker, start = start, end = end)
print(nifty)

In [ ]:
dailyReturns = nifty['Close'].pct_change(1) * 100
dailyReturns = dailyReturns.dropna()

In [ ]:
fig1 = plot_acf(dailyReturns, lags = 30)
fig2 = plot_pacf(dailyReturns, lags = 30, method = 'ywm')

In [ ]:
model = ARIMA(dailyReturns, order = (1, 0, 1))
model_fit = model.fit()

print(model_fit.summary())

### Why not difference the returns?

The ADF test already showed the **return series is stationary**, so the correct order of integration is `d = 0`. Differencing a stationary series (e.g. `d = 8`) does not help — it injects artificial structure, inflates the variance, and makes the MA term non-invertible (a quick test of `order=(1, 8, 1)` sends the AIC from ~600 to ~2100). For returns we search over `p` and `q` only and keep `d = 0`; `d = 1` is what you would use to model the *price level* directly.

In [ ]:
model = ARIMA(dailyReturns, order = (1, 0, 0))
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
model = ARIMA(dailyReturns, order = (0, 0, 1))
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
model = ARIMA(dailyReturns, order = (1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
model = ARIMA(dailyReturns, order = (0, 0, 0))
model_fit = model.fit()

print(model_fit.summary())

### Conclusion — the returns are white noise in the mean

Across every specification the Ljung-Box residual test (`Prob(Q)`) is non-significant, and the plain **ARIMA(0,0,0)** mean model is as good as anything more complex. The `(1,0,1)` fit posts a marginally lower AIC, but its AR (≈ −0.97) and MA (≈ +0.92) roots nearly cancel — a redundant, over-parameterised model rather than a genuinely better one.

**Takeaway:** daily Nifty returns carry no exploitable linear memory — the same weak-form-EMH result the OLS notebook reached. The structure worth modelling is in the *variance*, which is what the GARCH notebook does.